In [ ]:
!python -m pip install lightning
Training = True

In [ ]:
import torch
import os

if torch.cuda.is_available():
    accelerator = "cuda"
    torch.cuda.memory.empty_cache()
    root_dir = "/content"
else:
    accelerator = "cpu"
    root_dir = "."
device = torch.device(accelerator)

project_name = "DE_ENG_Translator_V2"
if not os.path.exists(f"{root_dir}/Checkpoints"):
    os.mkdir(f"{root_dir}/Checkpoints")
if not os.path.exists(f"{root_dir}/Checkpoints/{project_name}"):
    os.mkdir(f"{root_dir}/Checkpoints/{project_name}")


if not os.path.exists(f"{root_dir}/TensorBoard"):
    os.mkdir(f"{root_dir}/TensorBoard")
if not os.path.exists(f"{root_dir}/TensorBoard/{project_name}"):
    os.mkdir(f"{root_dir}/TensorBoard/{project_name}")
if not os.path.exists(f"{root_dir}/TensorBoard/{project_name}/Loss"):
    os.mkdir(f"{root_dir}/TensorBoard/{project_name}/Loss")
if not os.path.exists(f"{root_dir}/TensorBoard/{project_name}/Loss/train"):
    os.mkdir(f"{root_dir}/TensorBoard/{project_name}/Loss/train")
if not os.path.exists(f"{root_dir}/TensorBoard/{project_name}/Loss/validation"):
    os.mkdir(f"{root_dir}/TensorBoard/{project_name}/Loss/validation")

In [ ]:
import logging
import sys

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

formatter = logging.Formatter('%(name)s : %(levelname)s:%(levelno)s  %(asctime)s    %(message)s ')

info_handler = logging.FileHandler(f'{root_dir}/TranslatorTrainingMetrics.log') # default mode is already 'append'
info_handler.setLevel(logging.INFO)
info_handler.setFormatter(formatter)
logger.addHandler(info_handler)

# Should prevent output of the logger in the command line
console_log_level = 100
console = logging.StreamHandler(sys.stdout)
console.setLevel(console_log_level)
logger.addHandler(console)

# 1. Prepare the Data

In [ ]:
import numpy as np
import pandas as pd

data = pd.read_csv("deu.txt", sep = "\t", header = None, usecols = [0, 1])
data.head()

## 1.1 Tokenizer

In [ ]:
import re
import nltk
nltk.download("punkt_tab")
from nltk.tokenize import WordPunctTokenizer,RegexpTokenizer, word_tokenize as first_tokenizer
second_tokenizer = WordPunctTokenizer().tokenize
third_tokenizer = RegexpTokenizer(r"\d", gaps = False).tokenize

def Tokenizer(String):
    Tokens = first_tokenizer(String)

    # Whitespice is also a token I want to predict. Insert where neccessary in the list.
    Tokens2 = []
    for i, Token in enumerate(Tokens):
        Tokens2.append(Token)
        if (i != len(Tokens) - 1) and (not re.search(r".*\d+.*", Token)):
            Tokens2.append(" ")

     # WordPunctTokenize to get to split of all the "$", ".", ":" and similar. Does not take care of something like "1st" or "2nd"
    Tokens2 = [second_tokenizer(Token) if re.search(r".*\d+.*", Token) else Token for Token in Tokens]
    Tokens = []
    for Token in Tokens2:
        if type(Token) is str:
            Tokens.append(Token)
        else:
            Tokens.extend(Token)

    # RegexpTokenizer to split somethinglike "888" into ["8", "8", "8"]
    Tokens2 = [third_tokenizer(Token) if re.search(r"\d+", Token) else Token for Token in Tokens]
    Tokens = []
    for Token in Tokens2:
        if type(Token) is str:
            Tokens.append(Token)
        else:
            Tokens.extend(Token)
    del Tokens2
    return Tokens

## 1.2 Label Mapping

In [ ]:
Padding_Length = 300

In [ ]:
#### German
print("German")
Texts = data.iloc[:, 1].to_numpy().tolist()

print("Tokenizing Words")
Tokenized = [Tokenizer(Text) for Text in Texts]
del Texts

print("Find Unique Tokens")
Unique_Tokens = set( "\t\t".join(["\t\t".join(Sentence) for Sentence in Tokenized]).split("\t\t") )
del Tokenized

print("Create Label Mapping (and its inverse)")
German_LabelMapping_dict = {Token: i+1 for i, Token in enumerate(Unique_Tokens)}
German_LabelMapping_dict[""] = 0 # Padding Token
German_LabelMapping_dict["[END]"] = len(German_LabelMapping_dict) # Sentence End Token
German_LabelMapping_dict["[START]"] = len(German_LabelMapping_dict) # Sentence Start Token
German_vocabsize = len(German_LabelMapping_dict)
German_InverseMapping_dict = {value:key for key, value in German_LabelMapping_dict.items()}
del Unique_Tokens

print("Done")

def German_LabelMapping(String, Padding_Length = Padding_Length, is_Input_Sequence = False):
    Tokens = Tokenizer(String)
    Mapped = [German_LabelMapping_dict["[START]"]] + [German_LabelMapping_dict[Token] for Token in Tokens]
    if is_Input_Sequence:
        Mapped = Mapped + [German_LabelMapping_dict["[END]"]]
    Diff = Padding_Length - len(Mapped)
    if Diff < 0:
        raise ValueError(f"Increase the Padding_Length. There is at least one mapped sentece that is longer than {Padding_Length} tokens.")
    else:
        Mapped.extend([German_LabelMapping_dict[""]]*Diff)
    return Mapped


In [ ]:
#### English
print("English")
Texts = data.iloc[:, 0].to_numpy().tolist()

print("Tokenizing Words")
Tokenized = [Tokenizer(Text) for Text in Texts]
print(Tokenized[:5])
del Texts

print("Find Unique Tokens")
Unique_Tokens = set( "\t\t".join(["\t\t".join(Sentence) for Sentence in Tokenized]).split("\t\t") )
del Tokenized

print("Create Label Mapping (and its inverse)")
English_LabelMapping_dict = {Token: i+1 for i, Token in enumerate(Unique_Tokens)}
English_LabelMapping_dict[""] = 0 # Padding Token
English_LabelMapping_dict["[END]"] = len(English_LabelMapping_dict) # Sentence End Token
English_LabelMapping_dict["[START]"] = len(English_LabelMapping_dict) # Sentence Start Token
English_vocabsize = len(English_LabelMapping_dict)
English_InverseMapping_dict = {value:key for key, value in English_LabelMapping_dict.items()}
del Unique_Tokens

print("Done")

def English_LabelMapping(String, Padding_Length = Padding_Length, is_Input_Sequence = False):
    Tokens = Tokenizer(String)
    Mapped = [English_LabelMapping_dict["[START]"]] + [English_LabelMapping_dict[Token] for Token in Tokens]
    if is_Input_Sequence:
        Mapped = Mapped + [English_LabelMapping_dict["[END]"]]
    Diff = Padding_Length - len(Mapped)
    if Diff < 0:
        raise ValueError(f"Increase the Padding_Length. There is at least one mapped sentece that is longer than {Padding_Length} tokens.")
    else:
        Mapped.extend([English_LabelMapping_dict[""]]*Diff)
    return Mapped


In [ ]:
vocab_size_dict = {"German": German_vocabsize, "English": English_vocabsize}

## 1.3 DataSet and DataLoader

In [ ]:
#### Create Dataset and DataLoader
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

class TranslatorDataset(Dataset):
    """
    German is the Input Language (Want to translate)
    English is the OutPut Language (the translation)
    """
    def __init__(self, X, y):
        self.German = X
        self.English = y
    def __len__(self):
        return len(self.German)
    def __getitem__(self, idx):
        German = np.array(German_LabelMapping(self.German[idx], is_Input_Sequence = True))
        English = np.array(English_LabelMapping(self.English[idx], is_Input_Sequence = False))
        return torch.tensor(German, dtype = torch.long).squeeze(0), torch.tensor(English, dtype = torch.long).squeeze(0)

X_train, X_val, y_train, y_val = train_test_split(data.iloc[:, 1].to_numpy().tolist(), data.iloc[:, 0].to_numpy().tolist(), test_size = 1/5)
# End Results: Train : Val = 80 : 20

Train_Set = TranslatorDataset(X_train, y_train)
Val_Set = TranslatorDataset(X_val, y_val)

batch_size = 20
# workers = 11
Train_Loader = DataLoader(Train_Set, batch_size = batch_size, shuffle = True, pin_memory=False)
Val_Loader = DataLoader(Val_Set, batch_size = batch_size, shuffle = False, pin_memory=False)

del Train_Set
del Val_Set
del X_train, X_val, y_train, y_val

# 2. Architecture

In [ ]:
import torch.nn as nn
import pytorch_lightning as pl
from pytorch_lightning import Trainer, callbacks
from pytorch_lightning.loggers import TensorBoardLogger
import datetime
import gc


class Masked_Loss(nn.Module):
    def __init__(self):
        super().__init__()
        self.LogSoftmax = nn.LogSoftmax(dim = -1)
        self.CrossEntropy = nn.CrossEntropyLoss(reduction = "none")

    def forward(self, predictions, targets):
        targets = targets[:, 1:] # shift to the right, get rid of the "['Start']" Token
        predictions = predictions[:, :-1, :] # make same length as the targets vecotr. We only loose a padding token
        padding_mask = targets != 0
        modified_targets = targets.clone()
        modified_targets[:, torch.sum(padding_mask, dim = 1)] = English_LabelMapping_dict["[END]"] # replace the first padding token with the "[End]" Token. Now the tensors should be identical IF the translation is perfect
        loss = torch.mean(self.CrossEntropy(predictions.permute(0, 2, 1), modified_targets) * padding_mask)
        return loss

class Masked_Acc(nn.Module):
    def __init__(self, OutputLanguage):
        super().__init__()
        self.LogSoftmax = nn.LogSoftmax(dim = -1)
        self.OutputLanguage = OutputLanguage

    def forward(self, predictions, targets):
        targets = targets[:, 1:] # shift to the right, get rid of the "['Start']" Token
        predictions = predictions[:, :-1, :]
        padding_mask = targets != 0
        modified_targets = targets.clone()
        modified_targets[:, torch.sum(padding_mask, dim = 1)] = English_LabelMapping_dict["[END]"]
        pred = torch.argmax(self.LogSoftmax(predictions), dim = -1)
        pred = torch.maximum(torch.minimum(pred, torch.full_like(pred, vocab_size_dict[self.OutputLanguage]-1)), torch.zeros_like(pred))
        match = modified_targets == pred
        match = torch.logical_and(match, padding_mask).type(torch.float)
        acc = torch.mean(match)
        return acc

class LearnedPositionalEmbedding(nn.Module):
    """
    From: https://medium.com/@benjybo7/unleash-the-power-of-positional-embeddings-5-techniques-and-how-to-implement-them-in-pytorch-8fc15d886c70
    A very simple positional Embedding
    """
    def __init__(self, seq_len, d_model):
        super().__init__()
        self.position_embeddings = nn.Embedding(seq_len, d_model)

    def forward(self, input_ids):
        positions = torch.arange(0, input_ids.size(1), device=input_ids.device).unsqueeze(0)
        return self.position_embeddings(positions)



class CustomLoggerCallback(pl.Callback):
    def on_train_epoch_end(self, trainer, pl_module):
        epoch_mean = torch.stack(pl_module.training_step_outputs_loss).mean()
        logger.info(f"Epoch {pl_module.current_epoch:0>3} Train Loss: {float(epoch_mean):.10f}")
        # free up the memory
        pl_module.training_step_outputs_loss.clear()

    def on_validation_epoch_end(self, trainer, pl_module):
        epoch_mean = torch.stack(pl_module.validation_step_outputs_loss).mean()
        logger.info(f"Epoch {pl_module.current_epoch:0>3} Valid Loss: {float(epoch_mean):.10f}")
        # free up the memory
        pl_module.validation_step_outputs_loss.clear()

        epoch_mean = torch.stack(pl_module.validation_step_outputs_accuary).mean()
        logger.info(f"Epoch {pl_module.current_epoch:0>3} Valid Accu: {float(epoch_mean)*100:.4f}")
        # free up the memory
        pl_module.validation_step_outputs_accuary.clear()


class TranslatorTransformerLightning(pl.LightningModule):
    def __init__(self, InputLanguage, OutputLanguage, root_dir,  project_name, embedded_size, n_heads, depth):
        super().__init__()

        self.InputLanguage = InputLanguage
        self.OutputLanguage = OutputLanguage
        self.checkpoint_path = root_dir + "/Checkpoints/" + project_name
        self.save_name = f"{project_name}_{self.InputLanguage}_to_{self.OutputLanguage}"

        self.InputEmbedding_Encoder = nn.Embedding(vocab_size_dict[self.InputLanguage], embedded_size)
        self.PositionalEmbedding_Encoder = LearnedPositionalEmbedding(vocab_size_dict[self.InputLanguage], embedded_size)
        EncoderLayer = nn.TransformerEncoderLayer(d_model = embedded_size, nhead = n_heads)
        self.Encoder = nn.TransformerEncoder(EncoderLayer, num_layers = depth)

        self.InputEmbedding_Decoder = nn.Embedding(vocab_size_dict[self.OutputLanguage], embedded_size)
        self.PositionalEmbedding_Decoder = LearnedPositionalEmbedding(vocab_size_dict[self.OutputLanguage], embedded_size)
        DecoderLayer = nn.TransformerDecoderLayer(d_model = embedded_size, nhead = n_heads)
        self.Decoder = nn.TransformerDecoder(DecoderLayer, num_layers = depth)

        multiplier1 = 5
        # multiplier2 = 10
        fc1 = nn.Linear(embedded_size, embedded_size*multiplier1)
        # fc2 = nn.Linear(embedded_size*multiplier1, embedded_size*multiplier1*multiplier2)
        fc3 = nn.Linear(embedded_size*multiplier1, vocab_size_dict[self.OutputLanguage])
        relu = nn.ReLU()
        norm = nn.BatchNorm1d(num_features = Padding_Length)

        self.Linear = nn.Sequential(
            fc1, norm, relu,
            # fc2, norm, relu,
            fc3
        )

        self.custom_loss = Masked_Loss()
        self.custom_acc = Masked_Acc(self.OutputLanguage)

        self.training_step_outputs_loss = []
        self.validation_step_outputs_loss = []
        self.validation_step_outputs_accuary = []

        # # Important: This property activates manual optimization.
        # self.automatic_optimization = False
        # self.optimizer = torch.optim.Adam(self.parameters(), lr=1e-4)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=1e-5)
        return optimizer

    def training_step(self, batch, batch_idx):
        """
        Implement a single training period with loss function. Training loss is logged (by default to Tensorboard)
        """
        Input, Output = batch
        Input_PaddingMask, Output_PaddingMask = (Input == 0).transpose(0, 1), (Output == 0).transpose(0, 1)
        # y_onehot = nn.functional.one_hot(y, num_classes = self.num_classes)
        x = self.InputEmbedding_Encoder(Input) + self.PositionalEmbedding_Encoder(Input)
        x = self.Encoder(x, src_key_padding_mask = Input_PaddingMask) # Wo kommt di ePaddingmask hin?

        # Decoder
        y = self.InputEmbedding_Decoder(Output) + self.PositionalEmbedding_Decoder(Output)
        y = self.Decoder(y, x, tgt_key_padding_mask = Output_PaddingMask) # Wo kommt di ePaddingmask hin?

        # Linear Layer
        logits = self.Linear(y)
        del x, y

        loss = self.custom_loss(logits, Output) # torch.nn.functional.cross_entropy(logits.type(torch.float), nn.functional.one_hot(Output).type(torch.float))
        # Logging to TensorBoard (if installed) by default
        self.log("train_loss", loss)
        self.training_step_outputs_loss.append(loss.detach())
        return loss

    def validation_step(self, batch, batch_idx):
        """
        Implement a single validation period with loss function. Validation loss is logged (by default to Tensorboard)
        """
        Input, Output = batch
        Input_PaddingMask, Output_PaddingMask = (Input == 0).transpose(0, 1), (Output == 0).transpose(0, 1)
        x = self.InputEmbedding_Encoder(Input) + self.PositionalEmbedding_Encoder(Input)
        x = self.Encoder(x, src_key_padding_mask = Input_PaddingMask)

        # Decoder
        y = self.InputEmbedding_Decoder(Output) + self.PositionalEmbedding_Decoder(Output)
        y = self.Decoder(y, x, tgt_key_padding_mask = Output_PaddingMask)

        # Linear Layer
        logits = self.Linear(y)
        del x, y

        loss = self.custom_loss(logits, Output) # torch.nn.functional.cross_entropy(logits.type(torch.float), nn.functional.one_hot(Output).type(torch.float))
        acc = self.custom_acc(logits, Output)
        # Logging to TensorBoard (if installed) by default
        self.log("val_loss", loss)
        self.log("val_acc", loss)
        self.validation_step_outputs_loss.append(loss.detach())
        self.validation_step_outputs_accuary.append(acc)

    def forward(self, sentence, Padding_Length = Padding_Length):
        """
        Forward funciton for 'predcition' when training is over
        """
        Input = torch.tensor(German_LabelMapping(sentence)).unsqueeze(0)
        Input_PaddingMask = torch.tensor(Input == 0).transpose(1, 0)
        Output = [English_LabelMapping_dict["[START]"]] + [0] * (Padding_Length-1)
        Output = torch.tensor(Output).unsqueeze(0)

        with torch.no_grad():
            encoder_output = self.Encoder(self.InputEmbedding_Encoder(Input) + self.PositionalEmbedding_Encoder(Input), src_key_padding_mask = Input_PaddingMask)
            for i in range(Padding_Length):
                Output_PaddingMask = torch.tensor(Output == 0).transpose(1, 0)
                decoder_output = self.Decoder(tgt = self.InputEmbedding_Decoder(Output) + self.PositionalEmbedding_Decoder(Output), memory = encoder_output, tgt_key_padding_mask = Output_PaddingMask)
                logits = self.Linear(decoder_output)
                prediction = torch.argmax(logits[:, i, :], axis = -1)
                Output[0, i] = prediction
                if prediction == English_LabelMapping_dict["[END]"]:
                    break

        Output = torch.minimum(Output, torch.full_like(Output, vocab_size_dict[self.OutputLanguage])).squeeze()
        Tokens = [English_InverseMapping_dict[int(i)] for i in Output if English_InverseMapping_dict[int(i)] not in ["[START]", "[END]"]]
        Sentence = "".join(Tokens)
        return Sentence

In [ ]:
torch.cuda.empty_cache()
gc.collect()
!set PYTORCH_CUDA_ALLOC_CONF = garbage_collection_threshold:1.0

# 3. Training

In [1]:
if Training:
    from torch import nn
    from pathlib import Path
    from torch.utils.tensorboard import SummaryWriter

    epochs = 50
    embedded_size = 100
    n_heads = 5
    depth = 4
    gradient_accumulation = 50
    gradient_clip_val = 1.0
    earlyStopping_threshold = 5
    minimal_val_loss = 1e-10

    Translator_DE_EN = TranslatorTransformerLightning(InputLanguage = "German", OutputLanguage = "English", root_dir = root_dir,  project_name = project_name, embedded_size = embedded_size, n_heads = n_heads, depth = depth).to(device)
    # print(Translator_DE_EN.checkpoint_path)
    previous_checkpoints = [i for i in os.listdir(Translator_DE_EN.checkpoint_path) if i.endswith(".weights.ckpt")] # Encoder Checkpoints share a directory.
    change_starting_epoch = False
    if previous_checkpoints:
        initial_epoch = np.max([int(i.split("epoch=")[-1].replace(".weights.ckpt", "")) for i in previous_checkpoints if i.endswith(".weights.ckpt")])
        Translator_DE_EN = TranslatorTransformerLightning.load_from_checkpoint(f"{Translator_DE_EN.checkpoint_path}/{Translator_DE_EN.save_name}_epoch={initial_epoch}.weights.ckpt", InputLanguage = "German", OutputLanguage = "English", root_dir = root_dir,  project_name = project_name, embedded_size = embedded_size, n_heads = n_heads, depth = depth).to(device)
        change_starting_epoch = True
    else:
        initial_epoch = 0






    earlyStop = callbacks.EarlyStopping(monitor='val_loss', patience=earlyStopping_threshold, min_delta = minimal_val_loss)
    # checkPointTraining = callbacks.ModelCheckpoint(dirpath = Translator_DE_EN.checkpoint_path, filename = Translator_DE_EN.save_name +  "_{epoch}.weights.ckpt", monitor='val_loss', verbose=0, save_top_k = -1, save_weights_only=False)
    checkPointTraining2 = callbacks.ModelCheckpoint(dirpath = Translator_DE_EN.checkpoint_path, filename = Translator_DE_EN.save_name +  "_{epoch}.weights", monitor='val_loss', verbose=0, save_top_k = -1, save_weights_only=True)
    custom_logger = CustomLoggerCallback()

    tensorboard_callback = TensorBoardLogger(
        save_dir = Translator_DE_EN.checkpoint_path,
        name = Translator_DE_EN.save_name,
    )




    trainer = Trainer(callbacks=[earlyStop, checkPointTraining2, custom_logger],
                      logger = tensorboard_callback,
                      max_epochs = epochs,
                      accelerator = accelerator,
                      log_every_n_steps=1,
                      devices = 1,
                      precision="16-mixed", # Mixed Precision Training for Memory and thus speed optimization
                      accumulate_grad_batches=gradient_accumulation,
                      gradient_clip_val = gradient_clip_val
                     )

    if change_starting_epoch:
        trainer.fit_loop.epoch_progress.current.processed = initial_epoch + 1

    if ".encoder_earlystop" in previous_checkpoints or initial_epoch +1 == epochs:
        print("Model has already been fully trained.")
    else:
        trainer.fit(model = Translator_DE_EN, train_dataloaders = Train_Loader, val_dataloaders = Val_Loader)

    Path(f"{Translator_DE_EN.checkpoint_path}/.encoder_earlystop").touch()

NameError: name 'Training' is not defined

In [ ]:
previous_checkpoints = [i for i in os.listdir(Translator_DE_EN.checkpoint_path) if i.endswith(".weights.ckpt")] # Encoder Checkpoints share a directory.
if previous_checkpoints:
    epoch = np.max([int(i.split("epoch=")[-1].replace(".weights.ckpt", "")) for i in previous_checkpoints if i.endswith(".weights.ckp")])
    Translator_DE_EN.load_state_dict(torch.load(f"{Translator_DE_EN.checkpoint_path}/{Translator_DE_EN.save_name}_epoch={epoch}.weights.ckpt", weights_only=False, map_location=device), strict = False)
    print("Savestate loaded")
else:
    print("No savestates found")

In [ ]:
Translator_DE_EN("Guten Tag, ich bin froh Sie zu treffen.")